## Deploy em Real Time

Fiz a disponibilização do endpoints em https://dbc-0b9b6719-6175.cloud.databricks.com/serving-endpoints/modelo_propensao/invocations.
Para o meu modelo de machine learning que está no catalogo unity propensao_compra_modelo. Com isso faz um código que simule os dados de cliente chegando em tempo real e realize a previsão do modelo. Lembrando de manter as colunas do dataframe corretas. Um Exemplo abaixo.
            "idade_cliente": 34,
            "renda_mensal_k": 12.5,
            "tempo_medio_clique_segundos": 45.2,
            "media_interacoes_suporte": 1,
            "media_cupons_ativos": 2,
            "media_score_nps_cliente": 9.0,
            "media_dias_inatividade": 5,
            "total_gasto_acumulado_reais": 1500.0,
            "genero_cliente_m": 0,
            "regiao_cliente_nordeste": 0,
            "regiao_cliente_norte": 0,
            "regiao_cliente_sudeste": 1,
            "regiao_cliente_sul": 0

In [0]:
import requests
import json
import pandas as pd
import time
import random

# Configurar autenticação - obtém o token do contexto do notebook
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# URL do endpoint
endpoint_url = "https://dbc-0b9b6719-6175.cloud.databricks.com/serving-endpoints/modelo_propensao/invocations"

# Função para gerar dados de cliente simulados
def gerar_cliente_aleatorio():
    return {
        "idade_cliente": random.randint(18, 30),
        "renda_mensal_k": round(random.uniform(1500.0, 25000.0), 2),
        "tempo_medio_clique_segundos": round(random.uniform(10.0, 120.0), 2),
        "media_interacoes_suporte": random.randint(0, 10),
        "media_cupons_ativos": random.randint(0, 5),
        "media_score_nps_cliente": round(random.uniform(0.0, 10.0), 1),
        "media_dias_inatividade": random.randint(0, 30),
        "total_gasto_acumulado_reais": round(random.uniform(100.0, 5000.0), 2),
        "genero_cliente_m": random.choice([0, 1]),
        "regiao_cliente_nordeste": 0,
        "regiao_cliente_norte": 0,
        "regiao_cliente_sudeste": 0,
        "regiao_cliente_sul": 0
    }

# Função para definir região (apenas uma pode ser 1)
def definir_regiao(cliente):
    regioes = ["regiao_cliente_nordeste", "regiao_cliente_norte", "regiao_cliente_sudeste", "regiao_cliente_sul"]
    regiao_selecionada = random.choice(regioes)
    cliente[regiao_selecionada] = 1
    return cliente

# Função para fazer previsão
def fazer_previsao(dados_cliente):
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    # Preparar payload - o endpoint espera um DataFrame serializado
    payload = {
        "dataframe_records": [dados_cliente]
    }
    
    try:
        response = requests.post(endpoint_url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Erro na requisição: {e}")
        if hasattr(e.response, 'text'):
            print(f"Resposta: {e.response.text}")
        return None

# Simular chegada de dados em tempo real
print("=" * 80)
print("SIMULAÇÃO DE PREVISÕES EM TEMPO REAL")
print("=" * 80)
print()

resultados = []

# Simular 5 clientes chegando
for i in range(1, 6):
    print(f"\n{'='*80}")
    print(f"Cliente {i} - Dados recebidos em tempo real")
    print(f"{'='*80}")
    
    # Gerar dados do cliente
    cliente = gerar_cliente_aleatorio()
    cliente = definir_regiao(cliente)
    
    # Exibir dados do cliente
    df_cliente = pd.DataFrame([cliente])
    print("\nDados do cliente:")
    print(df_cliente.T)
    
    # Fazer previsão
    print("\n🔄 Enviando para o modelo...")
    previsao = fazer_previsao(cliente)
    
    if previsao:
        print("\n✅ Previsão recebida:")
        print(json.dumps(previsao, indent=2))
        
        # Armazenar resultado
        resultado = cliente.copy()
        resultado['previsao'] = previsao.get('predictions', [None])[0]
        resultados.append(resultado)
    else:
        print("\n❌ Erro ao obter previsão")
    
    # Simular delay entre chegadas de dados
    if i < 5:
        print("\n⏳ Aguardando próximo cliente...")
        time.sleep(2)

# Exibir resumo
if resultados:
    print("\n" + "="*80)
    print("RESUMO DAS PREVISÕES")
    print("="*80)
    df_resultados = pd.DataFrame(resultados)
    display(df_resultados)
    
    # Estatísticas
    print("\n📊 Estatísticas:")
    if 'previsao' in df_resultados.columns:
        print(f"Total de clientes processados: {len(df_resultados)}")
        print(f"Previsões positivas (propensão a comprar): {df_resultados['previsao'].sum() if df_resultados['previsao'].dtype in ['int64', 'float64'] else 'N/A'}")